# Twitter (microblog)

> **Time-box:** 45–60 minutes. The follow graph and the timeline are where the depth lives — don't spend it on user CRUD.

## Core requirements

1. Users have a unique handle and a display name.
2. A user can post a short tweet (≤ 280 chars).
3. A user can follow / unfollow another user. Following is one-directional.
4. A user can fetch their **home timeline**: tweets from the people they follow, newest first, paginated.
5. A user can fetch any user's **profile timeline**: that user's own tweets.
6. A user can like / unlike a tweet. A tweet shows its like count.

## Stretch goals

- Retweets (a tweet with a reference to an original).
- Replies (threading).
- Mentions of `@handle` in the body — what does the schema need?
- Fan-out: at write-time (push to followers' inboxes) vs read-time (query at read).

## Things the interviewer will probe

- **Pagination:** offset vs cursor (`created_at + id`). What happens with new inserts mid-scroll?
- **Hot follows:** one user has 100M followers. Fan-out on write becomes catastrophic. How do you handle celebrity accounts?
- **Like counts:** denormalized counter on `tweets` vs live `COUNT(*)`? Trade-offs?
- **Composite keys:** `follows` is naturally `(follower_id, followee_id)`. Do you give it a synthetic `id` anyway? Why or why not?
- **Resource naming:** `POST /users/{handle}/follow` vs `POST /follows`? Defend a choice.

---
## Setup

In [1]:
import json
import sqlite3

import pandas as pd
from fastapi import FastAPI
from fastapi.testclient import TestClient
from IPython.display import display
from pydantic import BaseModel

## Schema

To generate the feed, we will:
1. Get everyone the user is following
2. Get their posts
3. Sort in reverse chronological order

`follows` will have columns `follower_id` and `followee_id`. Since the user is the follower, we should set the primary key as `(follower_id, followee_id)` (rather than the other way around) so that 1 can be performed efficiently.

2 requries us to join on `posts.user_id` and 3 requires us to sort in `posts.created_time`. Therefore, we can optimize these with an index `(user_id, created_time)` on `posts.

In [217]:
SCHEMA = """
CREATE TABLE users (
    id   INTEGER PRIMARY KEY,
    name TEXT NOT NULL
);

CREATE TABLE follows (
    follower_id INTEGER,
    followee_id INTEGER,
    PRIMARY KEY (follower_id, followee_id)
);

CREATE TABLE posts (
    id         INTEGER PRIMARY KEY,
    user_id    INTEGER NOT NULL REFERENCES users(id) ON DELETE CASCADE,
    content    TEXT    NOT NULL,
    created_at TEXT    NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_posts_user_created
    ON posts(user_id, created_at DESC);
"""

conn = sqlite3.connect(":memory:", check_same_thread=False)
conn.row_factory = sqlite3.Row
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript(SCHEMA)

print("Tables:", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()])

Tables: ['follows', 'posts', 'users']


## API

> After editing any cell below, re-run from **App** down through **Client**.

### Data model
It's clear that `user` should be a top-level resource and the endpoint would then be `/users`. It is also clear the `feed` should be a subresource of `user`, as the `feed` is a computed view of posts for a user and not an independent entity. The endpoint would therefore be `/users/{user_id}/feed`.

It is less clear for posts and follows.
A post is an independent entity that has its own ID, unlike follows, so you can make the argument that it should be a top-level resource. However, since we are only interesting in the "get feed" functionality, we will model post as a sub-resource of user for simplicity: `/users/{user_id}/posts`.

If follows are a sub-resource of user, should the endpoint be `following` or `followers`?
1. `/users/{follower_id}/following/{followee_id}`
2. `/users/{followee_id}/followers/{follower_id}`
It should be 1, because the `followee_id` is the `user_id` of the user making the request. It makes more sense to write to their resource, rather than that of the person they're following.

This would also make it simple to get everyone the user is following with `GET /users/{user_id}/following`, but less so to get a user's followers. If we were concerned about this functionality, we would probably have both endpoints.

A benefit of modelling follows as a top-level resource is that it would be simple to do either:
1. Get everyone `user123` is following as `GET /follows?follower_id=user123`
2. Get followers of `user123` as `GET /follows?followee_id=user123`

However, we would always want to call this URL with 1 of these 2 parameters, but both are optional. You could argue that this is poor syntax, though it's not a strong criticism of this data model.

Since we are only worried about generating feeds, let's model `following` as subresource of user.

##### Authentication
There is a JWT in the header, and we verify that it matches the `user_id` in the URLs.

In [238]:
# ── App + models ──────────────────────────────────────────────────────────────
app = FastAPI(title="Twitter")

class CreateUser(BaseModel):
    name: str

class CreatePost(BaseModel):
    content: str

In [219]:
# ── Users ─────────────────────────────────────────────────────────────────────
@app.get("/healthz")
def healthz():
    return {"status": "ok"}


@app.post("/users", status_code=201)
def create_user(payload: CreateUser):
    with conn:
        conn.execute("INSERT INTO users(name) VALUES (?)", (payload.name,))
    return {"status": "ok"}

#### PUT for idempotency
We use PUT rather than POST for a new follow for idempotency.
This is demonstrated below by duplicating calls to this endpoint  

In [220]:
# ── Follows ───────────────────────────────────────────────────────────────────
@app.put("/users/{user_id}/following/{followee_id}", status_code=200)
def follow(user_id: int, followee_id: int):
    with conn:
        conn.execute(
            "INSERT OR IGNORE INTO follows(follower_id, followee_id) VALUES (?, ?)",
            (user_id, followee_id),
        )
    return {"status": "ok"}

In [221]:
# ── Posts ─────────────────────────────────────────────────────────────────────
@app.post("/users/{user_id}/posts", status_code=201)
def create_post(user_id: int, payload: CreatePost):
    with conn:
        conn.execute(
            "INSERT INTO posts(user_id, content) VALUES (?, ?)",
            (user_id, payload.content),
        )
    return {"status": "ok"}

In [222]:
# ── Feeds / profile timeline ───────────────────────────────────────────────────
@app.get("/users/{user_id}/feed")
def get_feed(user_id: int):
    rows = conn.execute(
        """
        SELECT p.id, p.user_id, u.name, p.content, p.created_at
        FROM posts p
        JOIN users u ON u.id = p.user_id
        WHERE p.user_id IN (
            SELECT followee_id FROM follows WHERE follower_id = ?
        )
        ORDER BY p.created_at DESC
        """,
        (user_id,),
    ).fetchall()
    return {"feed": [dict(row) for row in rows]}


In [223]:
# ── Client ────────────────────────────────────────────────────────────────────
client = TestClient(app, raise_server_exceptions=True)
print(client.get("/healthz").json())

{'status': 'ok'}


## Helpers

In [224]:
def call(method: str, path: str, **kwargs):
    r = getattr(client, method)(path, **kwargs)
    body = r.json() if r.content else None
    print(f"{method.upper():6s} {path}  →  {r.status_code}")
    if body is not None:
        print(json.dumps(body, indent=2))
    return r


def df(table: str) -> pd.DataFrame:
    return pd.read_sql(f"SELECT * FROM {table}", conn)


def query(sql: str, *params) -> pd.DataFrame:
    return pd.read_sql(sql, conn, params=list(params) if params else None)


def show_all():
    names = [r[0] for r in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()]
    for name in names:
        count = conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
        print(f"\n── {name} ({count} rows) ──")
        display(pd.read_sql(f"SELECT * FROM {name}", conn))

## Demo

In [225]:
call("post", "/users", json={"name": "Alice"})
call("post", "/users", json={"name": "Bob"})
call("post", "/users", json={"name": "Carol"})
df("users")

POST   /users  →  201
{
  "status": "ok"
}
POST   /users  →  201
{
  "status": "ok"
}
POST   /users  →  201
{
  "status": "ok"
}


,id,name
0,1,Alice
1,2,Bob
2,3,Carol


In [226]:
call("put", "/users/1/following/2")   # Alice → Bob
call("put", "/users/1/following/3")   # Alice → Carol
call("put", "/users/2/following/1")   # Bob → Alice
# duplicate follows — idempotency: should return 200, rows unchanged
call("put", "/users/1/following/2")   # Alice → Bob (again)
call("put", "/users/2/following/1")   # Bob → Alice (again)
df("follows")

PUT    /users/1/following/2  →  200
{
  "status": "ok"
}
PUT    /users/1/following/3  →  200
{
  "status": "ok"
}
PUT    /users/2/following/1  →  200
{
  "status": "ok"
}
PUT    /users/1/following/2  →  200
{
  "status": "ok"
}
PUT    /users/2/following/1  →  200
{
  "status": "ok"
}


,follower_id,followee_id
0,1,2
1,1,3
2,2,1


In [227]:
call("post", "/users/2/posts", json={"content": "Hello from Bob!"})
call("post", "/users/3/posts", json={"content": "Carol here. Hi everyone!"})
call("post", "/users/2/posts", json={"content": "Bob's second post."})
call("post", "/users/1/posts", json={"content": "Alice's first post."})
df("posts")

POST   /users/2/posts  →  201
{
  "status": "ok"
}
POST   /users/3/posts  →  201
{
  "status": "ok"
}
POST   /users/2/posts  →  201
{
  "status": "ok"
}
POST   /users/1/posts  →  201
{
  "status": "ok"
}


,id,user_id,content,created_at
0,1,2,Hello from Bob!,2026-05-24 12:55:12
1,2,3,Carol here. Hi everyone!,2026-05-24 12:55:12
2,3,2,Bob's second post.,2026-05-24 12:55:12
3,4,1,Alice's first post.,2026-05-24 12:55:12


In [237]:
r = call("get", "/users/1/feed")    # Alice's home feed (posts from Bob + Carol)
pd.json_normalize(r.json()["feed"])

GET    /users/1/feed  →  200
{
  "feed": [
    {
      "id": 3,
      "name": "Bob",
      "content": "Bob's second post.",
      "created_at": "2026-05-24 12:55:12"
    },
    {
      "id": 2,
      "name": "Carol",
      "content": "Carol here. Hi everyone!",
      "created_at": "2026-05-24 12:55:12"
    },
    {
      "id": 1,
      "name": "Bob",
      "content": "Hello from Bob!",
      "created_at": "2026-05-24 12:55:12"
    }
  ]
}


,id,name,content,created_at
0,3,Bob,Bob's second post.,2026-05-24 12:55:12
1,2,Carol,Carol here. Hi everyone!,2026-05-24 12:55:12
2,1,Bob,Hello from Bob!,2026-05-24 12:55:12


## Materialized view
We wish to minimize the latency in loading user feed, but the operation to get feed is expensive. Let's create a materialized view, which is a denormalized table representing the feed for users.

### Schema design
Each row is **one post in one user's feed**. A background job will run periodically and fan-out newly created posts to each follower's rows in this table.

**Finding a user's feed** is `WHERE user_id = ?`. The `user_id` column is the feed identifier — all rows sharing the same `user_id` constitute that user's feed.

**Primary key: `(user_id, post_id)`.**  A specific post should appear at most once in a specific user's feed. Making this the primary key:
- prevents duplicate rows if the background job re-runs (idempotent upserts with `INSERT OR IGNORE`)
- gives an efficient index for feed lookups, since `user_id` is the leading column

The denormalized columns (`poster_name`, `content`, `created_at`) are plain copies — they carry no relational constraint, so they get no `REFERENCES` clause.


In [229]:
conn.execute(
    """
    CREATE TABLE IF NOT EXISTS feeds (
        user_id     INTEGER NOT NULL REFERENCES users(id) ON DELETE CASCADE,
        post_id     INTEGER NOT NULL REFERENCES posts(id) ON DELETE CASCADE,
        poster_name TEXT    NOT NULL,
        content     TEXT    NOT NULL,
        created_at  TEXT    NOT NULL,
        PRIMARY KEY (user_id, post_id)
    )
    """
)
conn.commit()

print("Tables:", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()])

Tables: ['feeds', 'follows', 'posts', 'users']


In [230]:
def populate_feeds():
    """Fan out every post to each follower's feed (full refresh, idempotent)."""
    with conn:
        conn.execute(
            """
            INSERT OR IGNORE INTO feeds (user_id, post_id, poster_name, content, created_at)
            SELECT
                f.follower_id  AS user_id,
                p.id           AS post_id,
                u.name         AS poster_name,
                p.content,
                p.created_at
            FROM posts p
            JOIN users  u ON u.id = p.user_id
            JOIN follows f ON f.followee_id = p.user_id
            """
        )

In [231]:
# Remove the old live-query route so the materialized-view version takes over.
app.router.routes = [
    r for r in app.router.routes
    if not (hasattr(r, "path") and r.path == "/users/{user_id}/feed")
]


@app.get("/users/{user_id}/feed")
def get_feed(user_id: int):
    rows = conn.execute(
        """
        SELECT post_id AS id, poster_name AS name, content, created_at
        FROM feeds
        WHERE user_id = ?
        ORDER BY created_at DESC, post_id DESC
        """,
        (user_id,),
    ).fetchall()
    return {"feed": [dict(row) for row in rows]}

In [232]:
populate_feeds()
df("feeds")

,user_id,post_id,poster_name,content,created_at
0,1,1,Bob,Hello from Bob!,2026-05-24 12:55:12
1,1,3,Bob,Bob's second post.,2026-05-24 12:55:12
2,1,2,Carol,Carol here. Hi everyone!,2026-05-24 12:55:12
3,2,4,Alice,Alice's first post.,2026-05-24 12:55:12


In [235]:
r = call("get", "/users/1/feed")    # Alice's feed — now reads from materialized view
pd.json_normalize(r.json()["feed"])

GET    /users/1/feed  →  200
{
  "feed": [
    {
      "id": 3,
      "name": "Bob",
      "content": "Bob's second post.",
      "created_at": "2026-05-24 12:55:12"
    },
    {
      "id": 2,
      "name": "Carol",
      "content": "Carol here. Hi everyone!",
      "created_at": "2026-05-24 12:55:12"
    },
    {
      "id": 1,
      "name": "Bob",
      "content": "Hello from Bob!",
      "created_at": "2026-05-24 12:55:12"
    }
  ]
}


,id,name,content,created_at
0,3,Bob,Bob's second post.,2026-05-24 12:55:12
1,2,Carol,Carol here. Hi everyone!,2026-05-24 12:55:12
2,1,Bob,Hello from Bob!,2026-05-24 12:55:12


## All tables

In [234]:
show_all()


── feeds (4 rows) ──


,user_id,post_id,poster_name,content,created_at
0,1,1,Bob,Hello from Bob!,2026-05-24 12:55:12
1,1,3,Bob,Bob's second post.,2026-05-24 12:55:12
2,1,2,Carol,Carol here. Hi everyone!,2026-05-24 12:55:12
3,2,4,Alice,Alice's first post.,2026-05-24 12:55:12



── follows (3 rows) ──


,follower_id,followee_id
0,1,2
1,1,3
2,2,1



── posts (4 rows) ──


,id,user_id,content,created_at
0,1,2,Hello from Bob!,2026-05-24 12:55:12
1,2,3,Carol here. Hi everyone!,2026-05-24 12:55:12
2,3,2,Bob's second post.,2026-05-24 12:55:12
3,4,1,Alice's first post.,2026-05-24 12:55:12



── users (3 rows) ──


,id,name
0,1,Alice
1,2,Bob
2,3,Carol
